# E1 Stage 1 cycle S2-B1-c-r1 -- fail-closed baseline, fresh-server provenance

Code `/Users/terrancehamilton/reachy-1-2-sim-stage2` = `e6c30b74e09c38e616dee0819a7e205d12c5853e`. Legs: `S2-B1-c-r1-setup` PLACE_ROUTE via `rig_motion.deploy_to_rest` (HOME->REST), `S2-B1-c-r1-flight` LIFT_TO_PRESENT via `rig_motion.to_present` (REST->PRESENT). One attempt each; a `stop` marker, a failed start check, an invalid baseline `cmd_seq`, or a failed compliance/provenance check means no motion.

In [1]:
# Cell 1 -- connect with literals; hygiene shown; tree identity pinned at generation time
import os, subprocess, sys, time, json, pathlib, traceback
sys.path.insert(0, "/Users/terrancehamilton/reachy-1-2-sim-stage2/src"); sys.path.insert(0, "/Users/terrancehamilton/reachy-1-2-sim-stage2/scripts")
sys.path.insert(0, "/Users/terrancehamilton/reachy-1-2-sim-stage2/scripts/e1_stage1")
CTRL = pathlib.Path("/Users/terrancehamilton/e1-stage2-B1-s1-2026-09-19/control"); RECORD_ROOT = "/Users/terrancehamilton/e1-stage2-B1-s1-2026-09-19/e1_server_runs"
SCENE = "/Users/terrancehamilton/reachy-1-2-sim-stage2/scenes/e1_boards/B1_evidence.yaml"
LEAD_IN_S = 3.0; COMPLIANCE_TIMEOUT_S = 3.0; CYCLE = "S2-B1-c-r1"
REPO = "/Users/terrancehamilton/reachy-1-2-sim-stage2"; GENERATED_AT_SHA = "e6c30b74e09c38e616dee0819a7e205d12c5853e"
REQUIRED_SHA = "3d7fc744f81eae14c39b70f22fab3c95fd02626d"; MERGE_TIME_ISO = "2026-09-16T02:20:18Z"
_current_sha = subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True, timeout=5).stdout.strip()
assert _current_sha == GENERATED_AT_SHA, f"kernel tree {_current_sha} != generated-at tree {GENERATED_AT_SHA} -- regenerate the notebook"
print("REACHY env in this kernel:", {k: v for k, v in os.environ.items() if k.upper().startswith("REACHY")})
assert "REACHY_IP" not in os.environ and "REACHY_ENABLE_MOTION" not in os.environ, "shell hygiene violated"
from reachy_sdk import ReachySDK
HOST, PORT = "localhost", 50051
reachy = ReachySDK(host=HOST, sdk_port=PORT)
print(f"ReachySDK(host={HOST!r}, sdk_port={PORT}) connected at wall {time.time_ns()} mono {time.monotonic_ns()}")
print("python:", sys.executable)


REACHY env in this kernel: {}


ReachySDK(host='localhost', sdk_port=50051) connected at wall 1789809844958682000 mono 382210313399000
python: /Users/terrancehamilton/e1venv/bin/python


In [2]:
# Cell 2 -- motion-client binding check + W4 fresh-server provenance
import e1_identity, provenance, gating, plan
from reachy_ai.motion import rig_routes as R
from reachy_ai.motion import primitives
from reachy_ai.tasks import rig_motion
def _pose(): return {name: float(getattr(reachy.r_arm, name).present_position) for name in R.R_JOINTS}
ident = e1_identity.verify_simulator_identity(host=HOST, port=PORT, scene_path=SCENE, record_root=RECORD_ROOT, read_sdk_joints=_pose)
d = ident.as_dict()
manifest = {}
if ident.run_dir:
    manifest_path = pathlib.Path(ident.run_dir) / "manifest.json"
    if manifest_path.is_file():
        manifest = json.loads(manifest_path.read_text())
def _git_is_ancestor(a, b):
    return subprocess.run(["git", "-C", REPO, "merge-base", "--is-ancestor", a, b]).returncode == 0
prov_ok, prov_reasons = provenance.check_binding_provenance(manifest, git_is_ancestor=_git_is_ancestor, required_sha=REQUIRED_SHA, merge_time_iso=MERGE_TIME_ISO, generated_at_sha=GENERATED_AT_SHA)
d["provenance_ok"] = prov_ok; d["provenance_reasons"] = prov_reasons
print(json.dumps(d, indent=2, default=str))
BINDING_OK = bool(ident.ok) and prov_ok
(CTRL / (f"binding_ok_{CYCLE}" if BINDING_OK else f"binding_FAIL_{CYCLE}")).write_text(json.dumps(d, default=str))
print("BINDING_OK =", BINDING_OK); print("present:", {k: round(v, 1) for k, v in _pose().items()})
def wait_for(pred, timeout_s, period=0.25):
    return gating.wait_for(CTRL, pred, timeout_s, period)
def start_check(kind):
    p = _pose(); here = R.posture_of(p)
    if kind == "PLACE_ROUTE_start": ok, why = rig_motion.check_start(reachy.r_arm, R.PLACE_ROUTE)
    elif kind == "PRESENT": ok = R.at_pose(p, R.PRESENT, tol=12.0, joints=list(R.GROSS_JOINTS)); why = "" if ok else "not at PRESENT (gross, 12 deg)"
    elif kind == "REST": ok = R.at_pose(p, R.REST, tol=12.0, joints=list(R.GROSS_JOINTS)); why = "" if ok else "not at REST (gross, 12 deg)"
    return {"kind": kind, "ok": bool(ok), "why": why, "posture_of": here, "pose": {k: round(v, 1) for k, v in p.items()}}
PREV_OK = BINDING_OK
# Policy A per-cycle gate (decision note §4; PR #124 review M3): this
# cycle's parked-recording start-variant evidence must exist, be bound to
# THIS cycle, and independently recompute to stiff-zero from its own
# recorded pose. Missing, corrupt, wrong-cycle, or non-stiff-zero
# (including keyframe-sag) evidence prevents every leg's motion this
# cycle -- it can never be satisfied by another cycle's or another
# board's artifacts.
START_VARIANT_OK, START_VARIANT_REASON, START_VARIANT_DOC = plan.start_variant_gate(CTRL, CYCLE)
print("start_variant gate:", START_VARIANT_OK, START_VARIANT_REASON or "", START_VARIANT_DOC)
if not START_VARIANT_OK:
    (CTRL / f"binding_FAIL_{CYCLE}").write_text(json.dumps({"reason": "start_variant", "detail": START_VARIANT_REASON}, default=str))
# Fresh pre-cycle compliance gate (2026-09-16 re-review, R2): the posture-only
# stiff-zero classification above cannot tell "already stiff at reset" from
# "compliant and reading zero for a moment before it sags" -- read ONE fresh
# state sample (no min_cmd_seq: nothing has been commanded yet this cycle,
# freshness window only) and require every required joint's own `compliant`
# field is exactly False. Missing, malformed, stale, or explicitly compliant
# evidence refuses via require_compliance's own fail-closed contract, the
# same one the per-leg check after turn_on already relies on.
COMPLIANCE_CHECK = e1_identity.require_compliance(ident.run_dir, R.R_JOINTS, compliant=False, timeout_s=COMPLIANCE_TIMEOUT_S)
print("pre-cycle compliance gate:", COMPLIANCE_CHECK.ok, COMPLIANCE_CHECK.reasons)
if not COMPLIANCE_CHECK.ok:
    (CTRL / f"binding_FAIL_{CYCLE}").write_text(json.dumps({"reason": "pre_cycle_compliance", "detail": COMPLIANCE_CHECK.as_dict()}, default=str))
PREV_OK = PREV_OK and START_VARIANT_OK and COMPLIANCE_CHECK.ok


{
  "ok": true,
  "reasons": [],
  "run_dir": "/Users/terrancehamilton/e1-stage2-B1-s1-2026-09-19/e1_server_runs/run_20260919_090342",
  "manifest": {
    "format_version": 1,
    "started_at": "2026-09-19T09:03:42.202122+00:00",
    "code_sha": "e6c30b74e09c38e616dee0819a7e205d12c5853e",
    "code_sha_dirty": false,
    "model_path": "/Users/terrancehamilton/reachy-1-2-sim-stage2/native_mujoco/model/reachy_1_2.xml",
    "model_sha256": "618ef2499f6207d5e9e72f8b9a4e538b0521008afc1a80f3ecfbab22ab35abb4",
    "scene_path": "/Users/terrancehamilton/reachy-1-2-sim-stage2/scenes/e1_boards/B1_evidence.yaml",
    "scene_sha256": "f577fc47616ab1b4d68e0173c1adec6d1110039372f0f68bed31e7f7d6715346",
    "scene_revision": "initial",
    "mujoco_version": "3.11.0",
    "python_version": "3.14.0 (v3.14.0:ebf955df7a8, Oct  7 2025, 08:20:14) [Clang 16.0.0 (clang-1600.0.26.6)]",
    "platform": "macOS-26.6.2-arm64-arm-64bit-Mach-O",
    "protocol_version": 1,
    "calibration_provenance": "measured_202

In [3]:
# Leg S2-B1-c-r1-setup: PLACE_ROUTE via rig_motion.deploy_to_rest (HOME->REST) -- one attempt, gated on go_S2-B1-c-r1-setup + the recorder
LEG = {"leg": "S2-B1-c-r1-setup", "route": "PLACE_ROUTE", "tool": "rig_motion.deploy_to_rest", "cycle": CYCLE}
go = wait_for(lambda: (CTRL / "go_S2-B1-c-r1-setup").exists(), 1800) if PREV_OK else "not_eligible"
LEG["go"] = go; print("go:", go)
rec = wait_for(lambda: (CTRL / "recorder_S2-B1-c-r1-setup.log").exists() and "fly the route now" in (CTRL / "recorder_S2-B1-c-r1-setup.log").read_text(), 900) if go == "ready" else go
LEG["recorder_status"] = rec; print("recorder status:", rec)
if rec == "ready":
    time.sleep(LEAD_IN_S)
    LEG["start_check"] = start_check("PLACE_ROUTE_start"); print("start check:", LEG["start_check"])
if rec == "ready" and LEG["start_check"]["ok"]:
    LEG["t_start_mono_ns"] = time.monotonic_ns(); LEG["t_start_wall_ns"] = time.time_ns()
    phases = []
    baseline = e1_identity._read_last_state(pathlib.Path(ident.run_dir) / "states.jsonl")
    baseline_cmd_seq = (baseline or {}).get("cmd_seq")
    if isinstance(baseline_cmd_seq, bool) or not isinstance(baseline_cmd_seq, int):
        LEG["outcome"] = "STOP no_valid_baseline_cmd_seq"; LEG["baseline"] = repr(baseline_cmd_seq)
        (CTRL / "stop").write_text(json.dumps(LEG, default=str))
    else:
        reachy.turn_on("r_arm")
        chk = e1_identity.require_compliance(ident.run_dir, R.R_JOINTS, compliant=False,
                                             timeout_s=COMPLIANCE_TIMEOUT_S, min_cmd_seq=baseline_cmd_seq)
        LEG["compliance_check"] = chk.as_dict(); print("compliance:", chk.as_dict())
        if chk.ok:
            try:
                ret = rig_motion.deploy_to_rest(reachy.r_arm, on_phase=lambda *a: phases.append([time.monotonic_ns(), *map(str, a)]))
                LEG["outcome"] = "returned"; LEG["returned"] = ret
            except Exception as exc:
                LEG["outcome"] = f"EXC {type(exc).__name__}: {exc}"; traceback.print_exc()
        else:
            LEG["outcome"] = "STOP compliance_check"
            (CTRL / "stop").write_text(json.dumps(chk.as_dict()))
    LEG["phases"] = phases
    LEG["t_end_mono_ns"] = time.monotonic_ns(); LEG["t_end_wall_ns"] = time.time_ns()
    LEG["elapsed_s"] = (LEG["t_end_mono_ns"] - LEG["t_start_mono_ns"]) / 1e9
    LEG["end_pose"] = {k: round(v, 1) for k, v in _pose().items()}
    print("outcome:", LEG["outcome"], "elapsed %.1f s" % LEG["elapsed_s"]); print("returned:", LEG.get("returned")); print("end pose:", LEG["end_pose"])
else:
    LEG["outcome"] = "not_attempted"
PREV_OK = LEG["outcome"] == "returned"
(CTRL / "S2-B1-c-r1-setup_done").write_text(json.dumps(LEG, default=str)); print(json.dumps(LEG, default=str))


go: ready


recorder status: ready


start check: {'kind': 'PLACE_ROUTE_start', 'ok': True, 'why': '', 'posture_of': 'home', 'pose': {'r_shoulder_pitch': -0.0, 'r_shoulder_roll': 0.0, 'r_arm_yaw': -0.0, 'r_elbow_pitch': -0.0, 'r_forearm_yaw': -0.0, 'r_wrist_pitch': 0.0, 'r_wrist_roll': 0.1, 'r_gripper': -0.0}}
compliance: {'ok': True, 'reasons': [], 'per_joint': {'r_shoulder_pitch': {'compliant': False, 'effort': -8.221982682880232e-18, 'seq': 61369, 'sim_step': 19530}, 'r_shoulder_roll': {'compliant': False, 'effort': -0.05333187821664133, 'seq': 61369, 'sim_step': 19530}, 'r_arm_yaw': {'compliant': False, 'effort': -3.8438121129719953e-20, 'seq': 61369, 'sim_step': 19530}, 'r_elbow_pitch': {'compliant': False, 'effort': 5.836529286562399e-18, 'seq': 61369, 'sim_step': 19530}, 'r_forearm_yaw': {'compliant': False, 'effort': -4.79254398442643e-20, 'seq': 61369, 'sim_step': 19530}, 'r_wrist_pitch': {'compliant': False, 'effort': 1.291446676565242e-19, 'seq': 61369, 'sim_step': 19530}, 'r_wrist_roll': {'compliant': False, '

outcome: returned elapsed 33.6 s
returned: ['GRIP_SHUT', 'BACK', 'CURL', 'CURL_HIGH', 'TUCK', 'SWING_1', 'SWING_2', 'SWING_3', 'HOVER', 'REST_SHUT', 'REST']
end pose: {'r_shoulder_pitch': -40.0, 'r_shoulder_roll': -9.8, 'r_arm_yaw': 0.0, 'r_elbow_pitch': -45.3, 'r_forearm_yaw': -0.0, 'r_wrist_pitch': -9.8, 'r_wrist_roll': 30.0, 'r_gripper': -45.0}
{"leg": "S2-B1-c-r1-setup", "route": "PLACE_ROUTE", "tool": "rig_motion.deploy_to_rest", "cycle": "S2-B1-c-r1", "go": "ready", "recorder_status": "ready", "start_check": {"kind": "PLACE_ROUTE_start", "ok": true, "why": "", "posture_of": "home", "pose": {"r_shoulder_pitch": -0.0, "r_shoulder_roll": 0.0, "r_arm_yaw": -0.0, "r_elbow_pitch": -0.0, "r_forearm_yaw": -0.0, "r_wrist_pitch": 0.0, "r_wrist_roll": 0.1, "r_gripper": -0.0}}, "t_start_mono_ns": 382214959319083, "t_start_wall_ns": 1789809849604658000, "compliance_check": {"ok": true, "reasons": [], "per_joint": {"r_shoulder_pitch": {"compliant": false, "effort": -8.221982682880232e-18, "seq

In [4]:
# Leg S2-B1-c-r1-flight: LIFT_TO_PRESENT via rig_motion.to_present (REST->PRESENT) -- one attempt, gated on go_S2-B1-c-r1-flight + the recorder
LEG = {"leg": "S2-B1-c-r1-flight", "route": "LIFT_TO_PRESENT", "tool": "rig_motion.to_present", "cycle": CYCLE}
go = wait_for(lambda: (CTRL / "go_S2-B1-c-r1-flight").exists(), 1800) if PREV_OK else "not_eligible"
LEG["go"] = go; print("go:", go)
rec = wait_for(lambda: (CTRL / "recorder_S2-B1-c-r1-flight.log").exists() and "fly the route now" in (CTRL / "recorder_S2-B1-c-r1-flight.log").read_text(), 900) if go == "ready" else go
LEG["recorder_status"] = rec; print("recorder status:", rec)
if rec == "ready":
    time.sleep(LEAD_IN_S)
    LEG["start_check"] = start_check("REST"); print("start check:", LEG["start_check"])
if rec == "ready" and LEG["start_check"]["ok"]:
    LEG["t_start_mono_ns"] = time.monotonic_ns(); LEG["t_start_wall_ns"] = time.time_ns()
    phases = []
    baseline = e1_identity._read_last_state(pathlib.Path(ident.run_dir) / "states.jsonl")
    baseline_cmd_seq = (baseline or {}).get("cmd_seq")
    if isinstance(baseline_cmd_seq, bool) or not isinstance(baseline_cmd_seq, int):
        LEG["outcome"] = "STOP no_valid_baseline_cmd_seq"; LEG["baseline"] = repr(baseline_cmd_seq)
        (CTRL / "stop").write_text(json.dumps(LEG, default=str))
    else:
        reachy.turn_on("r_arm")
        chk = e1_identity.require_compliance(ident.run_dir, R.R_JOINTS, compliant=False,
                                             timeout_s=COMPLIANCE_TIMEOUT_S, min_cmd_seq=baseline_cmd_seq)
        LEG["compliance_check"] = chk.as_dict(); print("compliance:", chk.as_dict())
        if chk.ok:
            try:
                ret = rig_motion.to_present(reachy.r_arm, on_phase=lambda *a: phases.append([time.monotonic_ns(), *map(str, a)]))
                LEG["outcome"] = "returned"; LEG["returned"] = ret
            except Exception as exc:
                LEG["outcome"] = f"EXC {type(exc).__name__}: {exc}"; traceback.print_exc()
        else:
            LEG["outcome"] = "STOP compliance_check"
            (CTRL / "stop").write_text(json.dumps(chk.as_dict()))
    LEG["phases"] = phases
    LEG["t_end_mono_ns"] = time.monotonic_ns(); LEG["t_end_wall_ns"] = time.time_ns()
    LEG["elapsed_s"] = (LEG["t_end_mono_ns"] - LEG["t_start_mono_ns"]) / 1e9
    LEG["end_pose"] = {k: round(v, 1) for k, v in _pose().items()}
    print("outcome:", LEG["outcome"], "elapsed %.1f s" % LEG["elapsed_s"]); print("returned:", LEG.get("returned")); print("end pose:", LEG["end_pose"])
else:
    LEG["outcome"] = "not_attempted"
PREV_OK = LEG["outcome"] == "returned"
(CTRL / "S2-B1-c-r1-flight_done").write_text(json.dumps(LEG, default=str)); print(json.dumps(LEG, default=str))


go: ready


recorder status: ready


start check: {'kind': 'REST', 'ok': True, 'why': '', 'posture_of': 'rest', 'pose': {'r_shoulder_pitch': -40.0, 'r_shoulder_roll': -10.0, 'r_arm_yaw': -0.0, 'r_elbow_pitch': -45.3, 'r_forearm_yaw': -0.0, 'r_wrist_pitch': -9.8, 'r_wrist_roll': 30.0, 'r_gripper': -45.0}}
compliance: {'ok': True, 'reasons': [], 'per_joint': {'r_shoulder_pitch': {'compliant': False, 'effort': -0.28217509995769774, 'seq': 68915, 'sim_step': 94990}, 'r_shoulder_roll': {'compliant': False, 'effort': -0.1797508026797061, 'seq': 68915, 'sim_step': 94990}, 'r_arm_yaw': {'compliant': False, 'effort': 0.09943684087463242, 'seq': 68915, 'sim_step': 94990}, 'r_elbow_pitch': {'compliant': False, 'effort': 0.9547320385543969, 'seq': 68915, 'sim_step': 94990}, 'r_forearm_yaw': {'compliant': False, 'effort': 0.01335141162847852, 'seq': 68915, 'sim_step': 94990}, 'r_wrist_pitch': {'compliant': False, 'effort': -0.16792293859143292, 'seq': 68915, 'sim_step': 94990}, 'r_wrist_roll': {'compliant': False, 'effort': -0.0103312

outcome: returned elapsed 3.4 s
returned: ['PRESENT']
end pose: {'r_shoulder_pitch': -69.0, 'r_shoulder_roll': -24.8, 'r_arm_yaw': 0.1, 'r_elbow_pitch': -79.7, 'r_forearm_yaw': -0.0, 'r_wrist_pitch': 0.1, 'r_wrist_roll': -0.0, 'r_gripper': -45.0}
{"leg": "S2-B1-c-r1-flight", "route": "LIFT_TO_PRESENT", "tool": "rig_motion.to_present", "cycle": "S2-B1-c-r1", "go": "ready", "recorder_status": "ready", "start_check": {"kind": "REST", "ok": true, "why": "", "posture_of": "rest", "pose": {"r_shoulder_pitch": -40.0, "r_shoulder_roll": -10.0, "r_arm_yaw": -0.0, "r_elbow_pitch": -45.3, "r_forearm_yaw": -0.0, "r_wrist_pitch": -9.8, "r_wrist_roll": 30.0, "r_gripper": -45.0}}, "t_start_mono_ns": 382365886653916, "t_start_wall_ns": 1789810000533780000, "compliance_check": {"ok": true, "reasons": [], "per_joint": {"r_shoulder_pitch": {"compliant": false, "effort": -0.28217509995769774, "seq": 68915, "sim_step": 94990}, "r_shoulder_roll": {"compliant": false, "effort": -0.1797508026797061, "seq": 68

In [5]:
# Final read-only state; no further motion
print("final pose:", {k: round(v, 1) for k, v in _pose().items()}); print("done at wall", time.time_ns())


final pose: {'r_shoulder_pitch': -69.0, 'r_shoulder_roll': -24.8, 'r_arm_yaw': 0.1, 'r_elbow_pitch': -79.7, 'r_forearm_yaw': -0.0, 'r_wrist_pitch': 0.1, 'r_wrist_roll': -0.0, 'r_gripper': -45.0}
done at wall 1789810003927841000
